In [1]:
def analyze_transactions(transactions):
    seen = set()
    spending = {}
    for t in transactions:
        if t["id"] in seen:
            raise ValueError(f"Duplicate transaction id: {t['id']}")
        seen.add(t["id"])
        categories = spending.setdefault(t["customer"], {})
        if t["status"] == "success":
            categories[t["category"]] = categories.get(t["category"], 0) + t["amount"]
    report = []
    for customer, categories in spending.items():
        top = max(sorted(categories), key=lambda c: categories[c]) if categories else None
        report.append({"customer": customer, "total": sum(categories.values()), "top_category": top})
    report.sort(key=lambda r: (-r["total"], r["customer"]))
    for position, row in enumerate(report, 1):
        row["rank"] = position
    return report


data = [
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
    {"id": 4, "customer": "Zain", "category": "Food", "amount": 300, "status": "failed"},
]
for row in analyze_transactions(data):
    print(row)

try:
    analyze_transactions(data + [{"id": 1, "customer": "Ali", "category": "Food", "amount": 10, "status": "success"}])
except ValueError as e:
    print(e)

{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}
{'customer': 'Zain', 'total': 0, 'top_category': None, 'rank': 3}
Duplicate transaction id: 1


In [2]:
def longest_unique_window(text):
    last_seen = {}
    start = 0
    best_start = 0
    best_length = 0
    for i, ch in enumerate(text):
        if ch in last_seen and last_seen[ch] >= start:
            start = last_seen[ch] + 1
        last_seen[ch] = i
        if i - start + 1 > best_length:
            best_length = i - start + 1
            best_start = start
    return text[best_start:best_start + best_length], best_start, best_length


for s in ["abcabcbb", "", "bbbbb", "pwwkew", "a b!a b"]:
    print(repr(s), longest_unique_window(s))

'abcabcbb' ('abc', 0, 3)
'' ('', 0, 0)
'bbbbb' ('b', 0, 1)
'pwwkew' ('wke', 2, 3)
'a b!a b' ('a b!', 0, 4)


In [3]:
def same_value(a, b):
    return isinstance(a, bool) == isinstance(b, bool) and a == b


def find_paths(data, target, path=""):
    result = []
    if path and same_value(data, target):
        result.append(path)
    if isinstance(data, dict):
        for key, value in data.items():
            new_path = f"{path}.{key}" if path else str(key)
            result.extend(find_paths(value, target, new_path))
    elif isinstance(data, list):
        for index, value in enumerate(data):
            result.extend(find_paths(value, target, f"{path}[{index}]"))
    return result


sample = {"a": [{"x": 5}, 5], "b": {"y": 5}}
print(find_paths(sample, 5))
print(find_paths({"a": True, "b": 1, "c": [1, True]}, 1))
print(find_paths({"a": True, "b": 1, "c": [1, True]}, True))

['a[0].x', 'a[1]', 'b.y']
['b', 'c[0]']
['a', 'c[1]']


In [4]:
import heapq


def plan_execution(tasks, dependencies):
    known = set(tasks)
    graph = {t: [] for t in known}
    indegree = {t: 0 for t in known}
    for before, after in dependencies:
        if before not in known or after not in known:
            raise ValueError(f"Unknown task in dependency: ({before}, {after})")
        graph[before].append(after)
        indegree[after] += 1
    ready = [t for t in known if indegree[t] == 0]
    heapq.heapify(ready)
    order = []
    while ready:
        current = heapq.heappop(ready)
        order.append(current)
        for nxt in graph[current]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                heapq.heappush(ready, nxt)
    if len(order) != len(known):
        stuck = sorted(t for t in known if t not in order)
        raise ValueError(f"Cycle detected involving: {', '.join(stuck)}")
    return order


print(plan_execution(["build", "test", "deploy"], [("build", "test"), ("test", "deploy")]))
print(plan_execution(["c", "b", "a"], []))

for bad_tasks, bad_deps in [
    (["a", "b"], [("a", "b"), ("b", "a")]),
    (["a", "b"], [("a", "z")]),
]:
    try:
        plan_execution(bad_tasks, bad_deps)
    except ValueError as e:
        print(e)

['build', 'test', 'deploy']
['a', 'b', 'c']
Cycle detected involving: a, b
Unknown task in dependency: (a, z)


In [5]:
def tokenize(expression):
    tokens = []
    i = 0
    while i < len(expression):
        ch = expression[i]
        if ch.isspace():
            i += 1
        elif ch.isdigit():
            j = i
            while j < len(expression) and expression[j].isdigit():
                j += 1
            tokens.append(int(expression[i:j]))
            i = j
        elif ch in "+-*/()":
            tokens.append(ch)
            i += 1
        else:
            raise ValueError(f"Invalid character: {ch!r}")
    return tokens


def evaluate(expression):
    tokens = tokenize(expression)
    if not tokens:
        raise ValueError("Empty expression")
    precedence = {"+": 1, "-": 1, "*": 2, "/": 2, "u": 3}
    numbers = []
    operators = []

    def apply_top():
        op = operators.pop()
        if op == "u":
            if not numbers:
                raise ValueError("Malformed expression")
            numbers.append(-numbers.pop())
            return
        if len(numbers) < 2:
            raise ValueError("Malformed expression")
        right = numbers.pop()
        left = numbers.pop()
        if op == "+":
            numbers.append(left + right)
        elif op == "-":
            numbers.append(left - right)
        elif op == "*":
            numbers.append(left * right)
        else:
            if right == 0:
                raise ZeroDivisionError("Division by zero")
            numbers.append(left / right)

    expect_operand = True
    for token in tokens:
        if isinstance(token, int):
            if not expect_operand:
                raise ValueError("Malformed expression")
            numbers.append(token)
            expect_operand = False
        elif token == "(":
            if not expect_operand:
                raise ValueError("Malformed expression")
            operators.append(token)
        elif token == ")":
            if expect_operand:
                raise ValueError("Malformed expression")
            while operators and operators[-1] != "(":
                apply_top()
            if not operators:
                raise ValueError("Mismatched parentheses")
            operators.pop()
        else:
            if expect_operand:
                if token == "-":
                    operators.append("u")
                    continue
                raise ValueError("Malformed expression")
            while operators and operators[-1] != "(" and precedence[operators[-1]] >= precedence[token]:
                apply_top()
            operators.append(token)
            expect_operand = True
    if expect_operand:
        raise ValueError("Malformed expression")
    while operators:
        if operators[-1] == "(":
            raise ValueError("Mismatched parentheses")
        apply_top()
    return float(numbers[0])


for expr in ["-(3 + 2) * 4 + 18 / 3", "2 + 3 * 4", "(2 + 3) * 4", "--5", "10 / (5 - 5)", "2 +", "(2 + 3", "2 $ 3", ""]:
    try:
        print(repr(expr), "->", evaluate(expr))
    except (ValueError, ZeroDivisionError) as e:
        print(repr(expr), "-> Error:", e)

'-(3 + 2) * 4 + 18 / 3' -> -14.0
'2 + 3 * 4' -> 14.0
'(2 + 3) * 4' -> 20.0
'--5' -> 5.0
'10 / (5 - 5)' -> Error: Division by zero
'2 +' -> Error: Malformed expression
'(2 + 3' -> Error: Mismatched parentheses
'2 $ 3' -> Error: Invalid character: '$'
'' -> Error: Empty expression


In [6]:
import itertools
from dataclasses import dataclass


@dataclass(frozen=True)
class Transaction:
    txn_id: int
    action: str
    amount: float
    balance: float


class Account:
    def __init__(self, account_id, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self.account_id = account_id
        self._balance = balance
        self._transactions = []
        self._ids = itertools.count(1)

    @property
    def balance(self):
        return self._balance

    @property
    def transactions(self):
        return tuple(self._transactions)

    @staticmethod
    def validate_amount(amount):
        if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount <= 0:
            raise ValueError("Amount must be a positive number")

    def check_funds(self, amount):
        if amount > self._balance:
            raise ValueError(f"Insufficient funds in account {self.account_id}")

    def apply(self, action, amount, sign):
        self._balance += sign * amount
        self._transactions.append(Transaction(next(self._ids), action, amount, self._balance))

    def deposit(self, amount):
        self.validate_amount(amount)
        self.apply("deposit", amount, 1)

    def withdraw(self, amount):
        self.validate_amount(amount)
        self.check_funds(amount)
        self.apply("withdraw", amount, -1)

    def statement(self):
        lines = [f"Statement for {self.account_id}"]
        for t in self._transactions:
            lines.append(f"#{t.txn_id} {t.action:<13} {t.amount:>10.2f} balance {t.balance:>10.2f}")
        lines.append(f"Current balance: {self._balance:.2f}")
        return "\n".join(lines)


class Bank:
    def __init__(self):
        self._accounts = {}

    def open_account(self, account_id, balance=0):
        if account_id in self._accounts:
            raise ValueError(f"Account {account_id} already exists")
        self._accounts[account_id] = Account(account_id, balance)
        return self._accounts[account_id]

    def get(self, account_id):
        if account_id not in self._accounts:
            raise KeyError(f"Account {account_id} does not exist")
        return self._accounts[account_id]

    def transfer(self, source_id, target_id, amount):
        source = self.get(source_id)
        target = self.get(target_id)
        if source is target:
            raise ValueError("Cannot transfer to the same account")
        Account.validate_amount(amount)
        source.check_funds(amount)
        source.apply("transfer_out", amount, -1)
        target.apply("transfer_in", amount, 1)


bank = Bank()
bank.open_account("A", 1000)
bank.open_account("B", 200)
bank.transfer("A", "B", 350)
print(bank.get("A").balance, bank.get("B").balance)
print(bank.get("A").statement())
print(bank.get("B").statement())

for action in [
    lambda: bank.transfer("A", "B", 5000),
    lambda: bank.transfer("A", "Z", 10),
    lambda: bank.get("A").deposit(-5),
    lambda: bank.get("A").withdraw(0),
]:
    try:
        action()
    except (ValueError, KeyError) as e:
        print("Rejected:", e)
print(bank.get("A").balance, bank.get("B").balance)

try:
    bank.get("A").transactions[0].amount = 1
except Exception as e:
    print("Immutable:", type(e).__name__)

650 550
Statement for A
#1 transfer_out      350.00 balance     650.00
Current balance: 650.00
Statement for B
#1 transfer_in       350.00 balance     550.00
Current balance: 550.00
Rejected: Insufficient funds in account A
Rejected: 'Account Z does not exist'
Rejected: Amount must be a positive number
Rejected: Amount must be a positive number
650 550
Immutable: FrozenInstanceError


In [7]:
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        if not name:
            raise ValueError("Name is required")
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        pass


class SalariedEmployee(Employee):
    def __init__(self, name, monthly_salary):
        super().__init__(name)
        if monthly_salary < 0:
            raise ValueError("Salary cannot be negative")
        self.monthly_salary = monthly_salary

    def calculate_pay(self):
        return self.monthly_salary


class HourlyEmployee(Employee):
    def __init__(self, name, hourly_rate, hours_worked):
        super().__init__(name)
        if hourly_rate < 0 or hours_worked < 0:
            raise ValueError("Rate and hours cannot be negative")
        self.hourly_rate = hourly_rate
        self.hours_worked = hours_worked

    def calculate_pay(self):
        regular = min(self.hours_worked, 40)
        overtime = max(self.hours_worked - 40, 0)
        return regular * self.hourly_rate + overtime * self.hourly_rate * 1.5


class CommissionEmployee(Employee):
    def __init__(self, name, base_pay, sales, rate):
        super().__init__(name)
        if base_pay < 0 or sales < 0 or rate < 0:
            raise ValueError("Base, sales and rate cannot be negative")
        self.base_pay = base_pay
        self.sales = sales
        self.rate = rate

    def calculate_pay(self):
        return self.base_pay + self.sales * self.rate


def payroll_summary(employees):
    lines = []
    total = 0
    for e in employees:
        pay = e.calculate_pay()
        total += pay
        lines.append(f"{e.name:<10} {type(e).__name__:<20} {pay:>10.2f}")
    lines.append(f"{'TOTAL':<31} {total:>10.2f}")
    return "\n".join(lines)


staff = [
    SalariedEmployee("Ahmed", 3000),
    HourlyEmployee("Bilal", 20, 45),
    CommissionEmployee("Hina", 500, 4000, 0.10),
]
print(payroll_summary(staff))

try:
    Employee("Test")
except TypeError as e:
    print("Cannot instantiate:", e)

Ahmed      SalariedEmployee        3000.00
Bilal      HourlyEmployee           950.00
Hina       CommissionEmployee       900.00
TOTAL                              4850.00
Cannot instantiate: Can't instantiate abstract class Employee without an implementation for abstract method 'calculate_pay'


In [8]:
import functools
import time


def ttl_cache(seconds):
    def decorator(func):
        cache = {}
        stats = {"hits": 0, "misses": 0}

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            try:
                hash(key)
            except TypeError:
                stats["misses"] += 1
                return func(*args, **kwargs)
            now = time.monotonic()
            if key in cache:
                value, expires_at = cache[key]
                if now < expires_at:
                    stats["hits"] += 1
                    return value
            stats["misses"] += 1
            value = func(*args, **kwargs)
            cache[key] = (value, now + seconds)
            return value

        wrapper.cache_info = lambda: dict(stats)
        wrapper.cache_clear = lambda: (cache.clear(), stats.update(hits=0, misses=0))
        return wrapper

    return decorator


@ttl_cache(0.5)
def add(x, y):
    return x + y


print(add(x=2, y=3), add.cache_info())
print(add(y=3, x=2), add.cache_info())
time.sleep(0.6)
print(add(x=2, y=3), add.cache_info())
print(add([1], [2]), add([1], [2]), add.cache_info())
print(add.__name__)

5 {'hits': 0, 'misses': 1}
5 {'hits': 1, 'misses': 1}


5 {'hits': 1, 'misses': 2}
[1, 2] [1, 2] {'hits': 1, 'misses': 4}
add


In [9]:
from datetime import datetime


def read_errors(path, stats=None):
    with open(path, encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            parts = [p.strip() for p in line.rstrip("\n").split("|", 2)]
            valid = len(parts) == 3 and parts[1] != ""
            if valid:
                try:
                    datetime.strptime(parts[0], "%Y-%m-%d %H:%M:%S")
                except ValueError:
                    valid = False
            if not valid:
                if stats is not None:
                    stats["malformed"] += 1
                continue
            timestamp, level, message = parts
            if level == "ERROR":
                yield line_number, timestamp, message


def summarize_errors(path):
    stats = {"malformed": 0}
    by_date = {}
    for _, timestamp, _ in read_errors(path, stats):
        day = timestamp[:10]
        by_date[day] = by_date.get(day, 0) + 1
    return by_date, stats["malformed"]


with open("app.log", "w", encoding="utf-8") as f:
    f.write("2026-11-07 10:30:00 | ERROR | Connection lost\n")
    f.write("2026-11-07 10:31:00 | INFO | Reconnected\n")
    f.write("this line is broken\n")
    f.write("2026-11-07 11:00:00 | ERROR | Disk full\n")
    f.write("2026-13-45 99:00:00 | ERROR | Bad date\n")
    f.write("2026-11-08 09:15:00 | ERROR | Timeout\n")
    f.write("\n")

for record in read_errors("app.log"):
    print(record)

print(summarize_errors("app.log"))

(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 11:00:00', 'Disk full')
(6, '2026-11-08 09:15:00', 'Timeout')
({'2026-11-07': 2, '2026-11-08': 1}, 3)


In [10]:
import csv


def reconcile_inventory(inventory_path, movements_path, updated_path, rejected_path):
    with open(inventory_path, newline="", encoding="utf-8") as f:
        inventory = {
            row["sku"].strip(): {"name": row["name"], "quantity": int(row["quantity"])}
            for row in csv.DictReader(f)
        }
    rejected = []
    with open(movements_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            sku = (row.get("sku") or "").strip()
            kind = (row.get("type") or "").strip().upper()
            try:
                units = int(row.get("units"))
            except (TypeError, ValueError):
                units = None
            reason = None
            if sku not in inventory:
                reason = "Unknown SKU"
            elif kind not in ("IN", "OUT"):
                reason = "Invalid movement type"
            elif units is None or units <= 0:
                reason = "Invalid units"
            elif kind == "OUT" and inventory[sku]["quantity"] < units:
                reason = "Insufficient stock"
            if reason:
                rejected.append({"sku": sku, "type": row.get("type"), "units": row.get("units"), "reason": reason})
                continue
            inventory[sku]["quantity"] += units if kind == "IN" else -units
    with open(updated_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["sku", "name", "quantity"])
        for sku, item in inventory.items():
            writer.writerow([sku, item["name"], item["quantity"]])
    with open(rejected_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["sku", "type", "units", "reason"])
        writer.writeheader()
        writer.writerows(rejected)
    return inventory, rejected


with open("inventory.csv", "w", newline="", encoding="utf-8") as f:
    f.write("sku,name,quantity\nP01,Screen,10\nP02,Battery,5\n")

with open("movements.csv", "w", newline="", encoding="utf-8") as f:
    f.write("sku,type,units\nP01,OUT,4\nP01,OUT,9\nP01,IN,2\nP99,IN,5\nP02,IN,0\nP02,MOVE,3\nP02,OUT,5\n")

final_inventory, rejected_rows = reconcile_inventory(
    "inventory.csv", "movements.csv", "updated_inventory.csv", "rejected_movements.csv"
)
print(open("updated_inventory.csv").read())
print(open("rejected_movements.csv").read())

sku,name,quantity
P01,Screen,8
P02,Battery,0

sku,type,units,reason
P01,OUT,9,Insufficient stock
P99,IN,5,Unknown SKU
P02,IN,0,Invalid units
P02,MOVE,3,Invalid movement type



In [11]:
import csv


def grade_for(average):
    if average >= 85:
        return "A"
    if average >= 70:
        return "B"
    if average >= 55:
        return "C"
    if average >= 40:
        return "D"
    return "F"


def sort_key(student_id):
    return (int(student_id) if student_id.isdigit() else float("inf"), student_id)


def build_gradebook(students_path, scores_path, report_path):
    with open(students_path, newline="", encoding="utf-8") as f:
        students = {row["student_id"].strip(): row["name"].strip() for row in csv.DictReader(f)}
    marks = {}
    rejected = []
    with open(scores_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            student_id = row["student_id"].strip()
            subject = row["subject"].strip()
            try:
                value = float(row["marks"])
            except (TypeError, ValueError):
                rejected.append((row, "Invalid marks"))
                continue
            if student_id not in students:
                rejected.append((row, "Unknown student ID"))
            elif not 0 <= value <= 100:
                rejected.append((row, "Marks out of range"))
            elif subject in marks.get(student_id, {}):
                rejected.append((row, "Duplicate student-subject entry"))
            else:
                marks.setdefault(student_id, {})[subject] = value
    rows = []
    for student_id, subjects in marks.items():
        average = round(sum(subjects.values()) / len(subjects), 2)
        rows.append({"student_id": student_id, "name": students[student_id], "average": average, "grade": grade_for(average)})
    rows.sort(key=lambda r: (-r["average"], sort_key(r["student_id"])))
    for position, row in enumerate(rows, 1):
        row["rank"] = position
    with open(report_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["rank", "student_id", "name", "average", "grade"])
        writer.writeheader()
        writer.writerows(rows)
    return rows, rejected


with open("students.csv", "w", newline="", encoding="utf-8") as f:
    f.write("student_id,name\n101,Ali\n102,Sara\n103,Hamza\n")

with open("scores.csv", "w", newline="", encoding="utf-8") as f:
    f.write(
        "student_id,subject,marks\n"
        "101,Math,90\n101,Physics,80\n101,English,85\n"
        "102,Math,95\n102,Physics,60\n"
        "103,Math,30\n103,Math,40\n103,English,150\n"
        "999,Math,50\n"
    )

report_rows, rejected_scores = build_gradebook("students.csv", "scores.csv", "report.csv")
print(open("report.csv").read())
for row, reason in rejected_scores:
    print("Rejected:", dict(row), "->", reason)

rank,student_id,name,average,grade
1,101,Ali,85.0,A
2,102,Sara,77.5,B
3,103,Hamza,30.0,F

Rejected: {'student_id': '103', 'subject': 'Math', 'marks': '40'} -> Duplicate student-subject entry
Rejected: {'student_id': '103', 'subject': 'English', 'marks': '150'} -> Marks out of range
Rejected: {'student_id': '999', 'subject': 'Math', 'marks': '50'} -> Unknown student ID


In [12]:
class ReversibleRange:
    def __init__(self, start, stop, step=1):
        for value in (start, stop, step):
            if not isinstance(value, int) or isinstance(value, bool):
                raise TypeError("start, stop and step must be integers")
        if step == 0:
            raise ValueError("step must not be zero")
        self.start = start
        self.stop = stop
        self.step = step

    def __len__(self):
        if self.step > 0:
            return max(0, (self.stop - self.start + self.step - 1) // self.step)
        return max(0, (self.start - self.stop - self.step - 1) // (-self.step))

    def __iter__(self):
        value = self.start
        if self.step > 0:
            while value < self.stop:
                yield value
                value += self.step
        else:
            while value > self.stop:
                yield value
                value += self.step

    def __reversed__(self):
        index = len(self) - 1
        while index >= 0:
            yield self.start + index * self.step
            index -= 1

    def __contains__(self, item):
        if not isinstance(item, int) or isinstance(item, bool):
            return False
        if self.step > 0:
            inside = self.start <= item < self.stop
        else:
            inside = self.stop < item <= self.start
        return inside and (item - self.start) % self.step == 0


r = ReversibleRange(2, 11, 3)
print(list(r), list(reversed(r)), len(r), 5 in r, 6 in r)

import itertools

checked = 0
for start, stop, step in itertools.product(range(-6, 7), range(-6, 7), [-3, -2, -1, 1, 2, 3]):
    mine = ReversibleRange(start, stop, step)
    real = range(start, stop, step)
    assert list(mine) == list(real)
    assert list(reversed(mine)) == list(reversed(real))
    assert len(mine) == len(real)
    assert all((x in mine) == (x in real) for x in range(-8, 9))
    checked += 1
print("Matched built-in range on", checked, "cases")

[2, 5, 8] [8, 5, 2] 3 True False
Matched built-in range on 1014 cases


In [13]:
from dataclasses import dataclass
from datetime import datetime


@dataclass(frozen=True)
class Booking:
    booking_id: str
    room: str
    start: datetime
    end: datetime


class Scheduler:
    def __init__(self):
        self._bookings = {}
        self._ids = set()

    def book(self, booking_id, room, start, end):
        if not isinstance(start, datetime) or not isinstance(end, datetime):
            raise TypeError("start and end must be datetime objects")
        if end <= start:
            raise ValueError("End time must be after start time")
        if booking_id in self._ids:
            raise ValueError(f"Duplicate booking ID: {booking_id}")
        new = Booking(booking_id, room, start, end)
        conflicts = [b for b in self._bookings.get(room, []) if new.start < b.end and b.start < new.end]
        if conflicts:
            return False, conflicts
        self._bookings.setdefault(room, []).append(new)
        self._ids.add(booking_id)
        return True, []

    def bookings_for(self, room):
        return sorted(self._bookings.get(room, []), key=lambda b: b.start)


def at(hour, minute=0):
    return datetime(2026, 11, 7, hour, minute)


scheduler = Scheduler()
print(scheduler.book("B1", "A", at(9), at(10)))
print(scheduler.book("B2", "A", at(10), at(11)))
accepted, conflicts = scheduler.book("B3", "A", at(9, 30), at(10, 30))
print(accepted, [c.booking_id for c in conflicts])
print(scheduler.book("B4", "B", at(9, 30), at(10, 30)))

for args in [("B5", "A", at(12), at(12)), ("B1", "A", at(13), at(14))]:
    try:
        scheduler.book(*args)
    except ValueError as e:
        print("Rejected:", e)

for b in scheduler.bookings_for("A"):
    print(b.booking_id, b.start.time(), b.end.time())

(True, [])
(True, [])
False ['B1', 'B2']
(True, [])
Rejected: End time must be after start time
Rejected: Duplicate booking ID: B1
B1 09:00:00 10:00:00
B2 10:00:00 11:00:00
